# Pretrained HRNet-W18 baseline with the existing loss

This is a controlled initialization experiment for vertebral-corner localization. It changes only the HRNet-W18 backbone initialization from random to ImageNet pretrained (`hrnet_w18.ms_aug_in1k`). Dataset, preprocessing, augmentation, architecture, CenterNet loss weights, optimizer, batch size, seed, validation, and checkpoint-selection protocol remain aligned with the 80-epoch scratch baseline.

The existing corner loss remains plain masked L1 with `hm/reg/corner = 1.0/1.0/0.5`. No scale-normalized loss is enabled in this experiment.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import shutil

DRIVE_PACKAGE = Path('/content/drive/MyDrive/SOS-Colab')
DATASET_ROOT = DRIVE_PACKAGE / 'dataset'
PROJECT_ROOT = Path('/content/Spine-Opportunistic-Screening')
LOCAL_RUN_PARENT = Path('/content/centernet_runs')
DRIVE_RUN_PARENT = Path('/content/drive/MyDrive/spine_centernet_runs_nih_lumos')
EXPERIMENT = 'centernet_hrnet_w18_imagenet_pretrained_existing_loss_coco_nih_lumos_seed20260627'

RESUME = False  # First run: False. Continue this exact experiment only: True.
RUN_OVERFIT_SMOKE = False
RUN_TEST_EVALUATION = False  # Keep False until the experiment is frozen.
VALIDATION_CHECKPOINT = 'best_usable_recall'
NUM_WORKERS = 2

if not DRIVE_PACKAGE.is_dir():
    raise FileNotFoundError(f'Upload SOS-Colab to {DRIVE_PACKAGE}')
if not DATASET_ROOT.is_dir():
    raise FileNotFoundError(f'Dataset not found at {DATASET_ROOT}')
if PROJECT_ROOT.exists():
    shutil.rmtree(PROJECT_ROOT)
shutil.copytree(DRIVE_PACKAGE, PROJECT_ROOT, ignore=shutil.ignore_patterns('dataset'))
%cd /content/Spine-Opportunistic-Screening
print('code:', PROJECT_ROOT)
print('dataset:', DATASET_ROOT)
print('experiment:', EXPERIMENT)

In [ ]:
!pip install -q -r requirement.txt
import platform, cv2, numpy as np, torch, timm
from src.workflows.train_pretrained_w18_existing_loss import (
    EXPERIMENT_NAME, EXPECTED_DATASET_FINGERPRINT, PRETRAINED_BACKBONE_ID,
)
assert EXPERIMENT == EXPERIMENT_NAME
assert PRETRAINED_BACKBONE_ID == 'hrnet_w18.ms_aug_in1k'
assert timm.__version__ == '1.0.26', timm.__version__
print('python', platform.python_version())
print('torch', torch.__version__, 'cuda', torch.version.cuda)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
print('timm', timm.__version__, '| pretrained weights:', PRETRAINED_BACKBONE_ID)
if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime before training.')

## Verify the frozen dataset

These checks reject changed annotations, missing image files, unexpected split counts, or the wrong dataset fingerprint before training starts.

In [ ]:
import json
from collections import Counter
from src.data.dataset_provenance import build_dataset_provenance

EXPECTED = {
    'train': {'images': 1788, 'annotations': 22076, 'nih_images': 257, 'lumos_images': 128},
    'val': {'images': 223, 'annotations': 2976, 'nih_images': 32, 'lumos_images': 16},
    'test': {'images': 223, 'annotations': 3060, 'nih_images': 32, 'lumos_images': 16},
}
for split, expected in EXPECTED.items():
    split_dir = DATASET_ROOT / split
    with (split_dir / '_annotations.keypoints.coco.json').open(encoding='utf-8') as file:
        coco = json.load(file)
    nih = [x for x in coco['images'] if x['file_name'].startswith('images/nih_chestxray14/')]
    lumos = [x for x in coco['images'] if x['file_name'].startswith('images/lumos_ap/')]
    assert len(coco['images']) == expected['images']
    assert len(coco['annotations']) == expected['annotations']
    assert len(nih) == expected['nih_images']
    assert len(lumos) == expected['lumos_images']
    missing = [x['file_name'] for x in coco['images'] if not (split_dir / x['file_name']).is_file()]
    assert not missing, f'{split}: {len(missing)} missing images; first={missing[:1]}'
    print(split, dict(Counter(x.get('source_dataset', 'unknown') for x in coco['images'])))
provenance = build_dataset_provenance(DATASET_ROOT)
assert provenance['fingerprint'] == EXPECTED_DATASET_FINGERPRINT, provenance['fingerprint']
print('dataset fingerprint:', provenance['fingerprint'])

## Preflight tensors and pretrained model

The train check downloads the explicit pretrained W18 weights, builds the unchanged CenterNet heads, and performs a GPU forward/loss pass before the long run.

In [ ]:
import subprocess, sys

for split in ('train', 'val', 'test'):
    command = [
        sys.executable, '-u', '-m', 'src.workflows.check_centernet_dataset',
        '--dataset-root', str(DATASET_ROOT), '--split', split,
        '--image-size', '1024', '--backbone', 'hrnet_w18',
        '--batch-size', '2', '--num-workers', '0', '--limit', '4',
        '--hm-weight', '1.0', '--reg-weight', '1.0', '--wh-weight', '0.5',
    ]
    if split == 'train':
        command += ['--model-forward', '--pretrained', '--device', 'cuda']
    subprocess.run(command, check=True)

## Optional short plumbing smoke test

Enable `RUN_OVERFIT_SMOKE` only to check that optimization and checkpoint writing work. Its metrics are not an experimental result.

In [ ]:
if RUN_OVERFIT_SMOKE:
    subprocess.run([
        sys.executable, '-u', '-m', 'src.train_centernet',
        '--dataset-root', str(DATASET_ROOT), '--output-dir', '/content/centernet_overfit',
        '--experiment-name', 'pretrained_w18_existing_loss_overfit_8',
        '--backbone', 'hrnet_w18', '--pretrained', '--input-size', '512',
        '--overfit-samples', '8', '--epochs', '3', '--batch-size', '2',
        '--num-workers', '0', '--lr', '1e-4', '--hm-weight', '1.0',
        '--reg-weight', '1.0', '--wh-weight', '0.5', '--peak-thresh', '0.10',
        '--eval-topk', '50', '--progress-every', '2', '--amp',
    ], check=True)
else:
    print('optional overfit smoke skipped')

## Restore and train

Keep `RESUME=False` for the first run. The dedicated workflow freezes all metric-affecting settings and refuses a scratch or incompatible resume checkpoint. Artifacts train on local Colab storage and are backed up to Drive after every epoch.

In [ ]:
local_run = LOCAL_RUN_PARENT / EXPERIMENT
drive_run = DRIVE_RUN_PARENT / EXPERIMENT
if RESUME:
    if not (drive_run / 'last.pt').is_file():
        raise FileNotFoundError(f'No resumable checkpoint at {drive_run / "last.pt"}')
    local_run.mkdir(parents=True, exist_ok=True)
    shutil.copytree(drive_run, local_run, dirs_exist_ok=True)
    print('restored:', drive_run)
elif local_run.exists() or drive_run.exists():
    raise FileExistsError('A run with this experiment name already exists. Set RESUME=True only if it is this exact pretrained experiment.')

command = [
    sys.executable, '-u', '-m', 'src.workflows.train_pretrained_w18_existing_loss',
    '--dataset-root', str(DATASET_ROOT), '--output-dir', str(LOCAL_RUN_PARENT),
    '--backup-dir', str(DRIVE_RUN_PARENT), '--num-workers', str(NUM_WORKERS),
]
resume_checkpoint = local_run / 'last.pt'
if RESUME:
    command += ['--resume-checkpoint', str(resume_checkpoint)]
print(' '.join(command))
subprocess.run(command, check=True)

## Verify the trained checkpoint contract

This prevents a run from being reported as the pretrained/existing-loss baseline if its initialization, data, or loss settings differ.

In [ ]:
from src.evaluate_centernet import load_checkpoint
from src.workflows.train_pretrained_w18_existing_loss import validate_resume_checkpoint

best_checkpoint = local_run / f'{VALIDATION_CHECKPOINT}.pt'
if not best_checkpoint.is_file():
    best_checkpoint = local_run / 'best_center_f1.pt'
if not best_checkpoint.is_file():
    raise FileNotFoundError(best_checkpoint)
validate_resume_checkpoint(best_checkpoint)
checkpoint = load_checkpoint(best_checkpoint, torch.device('cpu'))
initialization = checkpoint.get('model_initialization', {})
assert initialization.get('pretrained') is True, initialization
assert initialization.get('pretrained_backbone_id') == PRETRAINED_BACKBONE_ID, initialization
assert checkpoint['args']['hm_weight'] == 1.0
assert checkpoint['args']['reg_weight'] == 1.0
assert checkpoint['args']['wh_weight'] == 0.5
print('checkpoint:', best_checkpoint.name, '| epoch:', checkpoint['epoch'])
print('initialization:', initialization)
print('loss: existing masked L1 corner regression; weights 1.0/1.0/0.5')

## Evaluate on validation

Model selection stays on validation. Test evaluation remains disabled until this experiment and the later normalized-loss experiment are frozen.

In [ ]:
val_output = local_run / f'evaluation_val_{best_checkpoint.stem}'
subprocess.run([
    sys.executable, '-u', '-m', 'src.evaluate_centernet',
    '--evaluation-profile', 'research', '--checkpoint', str(best_checkpoint),
    '--dataset-root', str(DATASET_ROOT), '--split', 'val',
    '--output-dir', str(val_output), '--batch-size', '2', '--num-workers', str(NUM_WORKERS),
    '--peak-thresh', '0.10', '--topk', '50',
], check=True)
drive_val_output = drive_run / val_output.name
shutil.copytree(val_output, drive_val_output, dirs_exist_ok=True)
with (val_output / 'metrics.json').open(encoding='utf-8') as file:
    evaluation = json.load(file)
chain = next(row for row in evaluation['scorecard']['rows'] if row['model'] == 'spine_chain' and row['scope'] == 'overall_micro')
corner_fields = [
    'gt_vertebrae', 'center_f1_0.20d', 'corner_nme_mean', 'corner_nme_p95',
    'pck_0.05', 'pck_0.10', 'end_to_end_pck_0.10', 'usable_vertebra_recall',
]
display({name: chain.get(name) for name in corner_fields})
print('validation artifacts:', drive_val_output)

In [ ]:
if RUN_TEST_EVALUATION:
    test_output = local_run / f'evaluation_test_{best_checkpoint.stem}'
    subprocess.run([
        sys.executable, '-u', '-m', 'src.evaluate_centernet',
        '--evaluation-profile', 'research', '--checkpoint', str(best_checkpoint),
        '--dataset-root', str(DATASET_ROOT), '--split', 'test',
        '--output-dir', str(test_output), '--batch-size', '2',
        '--num-workers', str(NUM_WORKERS), '--peak-thresh', '0.10', '--topk', '50',
    ], check=True)
    shutil.copytree(test_output, drive_run / test_output.name, dirs_exist_ok=True)
else:
    print('test evaluation disabled; use validation during experiment development')